# ML-03 — Frame My Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Divyanshu10045/flyrank-assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

> Every number below comes out of a cell I ran in this notebook; the placeholders I could not
> measure are called out as guesses in the text.

This week I take the lane I picked in ML-02 and pin it to a concrete task type, a target I can actually
compute from the starter CSV, and one metric I'd defend. It's a framing notebook, not a modeling one — the
model comes in ML-08, on the full warehouse release. The one model I *do* train here is a deliberate probe,
in Section 5, to answer "would a rule have done just as well?"

## 0. What I'm carrying over from ML-02

Lane 2 — refresh / content opportunity scoring. The deliverable is a **ranked review queue with a reason
code**, for a content editor who can only look at so many pages a week. The ML-02 numbers, re-verified on the
same 30,000-row CSV: 54.2% of pages are marked declining (16,262), 13,152 of those still get ≥100 impressions
per 90 days, and that pool holds ~80M pooled 90-day impressions. So the pool is roughly 329 weeks of work at
the 40-pages-a-week capacity I guessed — and that 40 is a guess, nothing in the data measures it.

One thing ML-02 left hanging, which this notebook closes: **ML-02's queue points at pages that are already
declining.** That's a triage filter, not a prediction. An editor standing in front of 13,152 already-slipping
pages still wants to know which ones are *about to* slip. That pushes the target forward in time, and the
starter CSV is not natively arranged for a forward target — it's one snapshot. Section 2 is where I fix that,
and the fix costs me most of the columns, which is the most interesting thing I learned this week.

In [1]:
# Setup: climb to the repo root so the relative data path works from wherever this kernel starts.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw"):
        parent = os.path.dirname(os.getcwd())
        if parent == os.getcwd():
            raise FileNotFoundError("data/raw not found - run this from inside the repo clone")
        os.chdir(parent)

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
RAW_CSV = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(RAW_CSV)

print("loaded:", RAW_CSV)
print("shape:", df.shape, "| clients:", df["client_id"].nunique(),
      "| unique content_id:", df["content_id"].nunique())
print("these are pseudonymized ids; the slice carries no titles, URLs, queries or client names.")

loaded: data/raw/content_refresh_anonymized.csv
shape: (30000, 44) | clients: 32 | unique content_id: 30000
these are pseudonymized ids; the slice carries no titles, URLs, queries or client names.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking, implemented as score-then-sort.** Not classification. Not clustering. Not signal analysis.

The argument, and the evidence is in the cell below:

- **The decision is a cut by capacity, not by confidence.** The action is "an editor opens the top of the list
  and works down it." Nobody in that workflow asks "is this page's probability above 0.5?" The queue is
  truncated by how many pages a person can review, so the only thing that matters is the *order* and the
  quality of the first few dozen entries.
- **A classifier threshold is useless at this base rate.** 54.2% of the inventory is already flagged
  declining. A 0.5-threshold classifier hands an editor 16,262 pages — 407 weeks at 40/week. It tells them
  nothing they didn't already have.
- **The metrics live at the top of the list.** A random queue over the whole slice scores P@50 = 0.48 against
  a 0.542 base rate (Section 3 shows how noisy a single small-K draw is). Beating the base rate is table
  stakes; the only thing that matters is movement at the *top* of the ordering, which needs a ranking metric.
- **Clustering is the wrong tool for the queue.** It would tell me what kinds of pages exist. Useful later for
  the action taxonomy and reason codes, but it doesn't put page #1 at the top of anything.

The nuance I want on the record: I *am* training a classifier under the hood, because that's the easy way to
build a ranker — fit a binary outcome, sort by predicted probability. The task type describes the
**deliverable and the evaluation**, not the model class. What makes this a ranking project rather than a
classification project is that the 0.5 threshold never gets used and the only number I report is Precision@K.

In [2]:
# Why the deliverable can't be a yes/no prediction. The only label that ships with
# the starter data is trend_direction, and it is already true for more than half the inventory.
ref_label = df["trend_direction"].eq("down")
WEEKLY_REVIEW_CAPACITY = 40  # GUESS, carried over from ML-02. Nothing in the data measures it.
demand_pool = ref_label & (df["impressions_90d"] >= 100)

print(f"rows: {len(df):,} | clients: {df['client_id'].nunique()} | unique content_id: {df['content_id'].nunique()}")
print(f"a 0.5-threshold classifier flags: {int(ref_label.sum()):,} pages = {ref_label.mean() * 100:.1f}% of the inventory")
print(f"  at {WEEKLY_REVIEW_CAPACITY}/week that is {ref_label.sum() / WEEKLY_REVIEW_CAPACITY:,.0f} weeks of review")
print(f"the ML-02 demand pool (declining AND >=100 impressions/90d): {int(demand_pool.sum()):,}"
      f" = {demand_pool.sum() / WEEKLY_REVIEW_CAPACITY:,.0f} weeks")
print()
print("trend_direction distribution (the only outcome-like column in the starter CSV):")
print(df["trend_direction"].value_counts().to_string())
print()

# A random queue is the null model, and it is flat in K. That is the whole reason the
# metric has to be a top-of-list metric.
rng = np.random.default_rng(0)
shuffled = np.argsort(-rng.random(len(df)), kind="stable")
y_null = ref_label.to_numpy()
print("Precision@K for a RANDOM queue (one draw, so treat the small-K values as noisy):")
for k in (40, 50, 200, 1000, 5000):
    print(f"  P@{k:<5d} {y_null[shuffled[:k]].mean():.3f}")
print(f"  base rate {y_null.mean():.3f}  <- what a random queue converges to as K grows")

rows: 30,000 | clients: 32 | unique content_id: 30000
a 0.5-threshold classifier flags: 16,262 pages = 54.2% of the inventory
  at 40/week that is 407 weeks of review
the ML-02 demand pool (declining AND >=100 impressions/90d): 13,152 = 329 weeks

trend_direction distribution (the only outcome-like column in the starter CSV):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

Precision@K for a RANDOM queue (one draw, so treat the small-K values as noisy):
  P@40    0.475
  P@50    0.480
  P@200   0.530
  P@1000  0.524
  P@5000  0.534
  base rate 0.542  <- what a random queue converges to as K grows


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### The label that ships with the repo is a rule, not an outcome

`scripts/01_prepare_features.py` sets `is_declining_label = (trend_direction == "down")`. The data dictionary
says `trend_direction` is a threshold on `trend_pct`, and `trend_pct` is computed from `impressions_last_30d`
versus `impressions_prev_30d`. So the label is a **rule applied to the current window**, and the features
describe that same window. That's the framing-ml-problems failure mode exactly: a label that comes from
someone's rule means the model learns the rule.

**I checked the shipped pipeline for this, and didn't find it.** `scripts/01_prepare_features.py` sets
`is_declining_label = (trend_direction == "down")`, so the obvious failure mode here is a model that can see
the column its own label is defined from, and I went looking for it. `scripts/03_train_model.py` builds its
feature matrix from `MODEL_NUMERIC_FEATURES` and `MODEL_CATEGORICAL_FEATURES` in `scripts/ml_utils.py` and
nothing else, and neither list contains `trend_direction`, `trend_pct`, or any `*_last_30d` / `*_prev_30d`
column. `trend_direction` does appear in `ml_utils.py`, but in a separate list (`CATEGORICAL_COLUMNS`) that
the prepare step only uses to clean values on its way into the CSV. So on the evidence I can see, the
reference model is trend-free, and the AUC 0.750 in `outputs/model_report.md` is consistent with that. I read
the report and the scripts rather than re-running `scripts/run_all.py`, because that would overwrite
`outputs/` and `scripts/` is read-only for me — so I'm reasoning from the source, not from a reproduction,
and whoever built it will know things I can't see from here.

**What I can't rule out, and what I did about it.** Nearly every numeric feature in that list is a 90-day
aggregate — `log_impressions_90d`, `log_clicks_90d`, `days_with_impressions`, `ctr`, `avg_position` and the
rest — while the label is a rule on `impressions_last_30d` against `impressions_prev_30d`, and
`impressions_last_30d` is a third of `impressions_90d`. Those features therefore span part of the window the
label is measured over. Whether that matters isn't my call: for a 90-day export it may be entirely
reasonable, and it's a plausible reason the reference model sits at 0.750 rather than near chance. I'm not
claiming it inflated anything. I'm flagging it as a question I'd want answered before treating 0.750 as a
forecasting number rather than a ranking score — and it is the reason I chose to be stricter with my own
frame, building features from the prior 60 days against a label on the newest 30 and discarding 24 columns
in the process. Being stricter than the reference isn't a criticism of it. It's just the cost I decided to pay
so that I can say my features are strictly earlier than my outcome.

### What I do instead: split the one window in three

The starter CSV is a single snapshot, but it isn't a single window. It hands me three 30-day blocks:

| block | column | days back from export |
|---|---|---|
| oldest | `impressions_90d − impressions_prev_30d − impressions_last_30d` (recovered by subtraction) | 61–90 |
| prior | `impressions_prev_30d` | 31–60 |
| label | `impressions_last_30d` | 1–30 |

The subtraction has no negatives on any of the 30,000 rows, so the blocks are real and not an artifact.

- **Feature window** = oldest + prior (days 31–90 back). Every feature comes from here.
- **Label window** = the newest block (days 1–30 back). Nothing a model is allowed to see touches it.

Features are now strictly earlier than the outcome. That's as close to past→future as one snapshot gets, and
it's a real improvement on `is_declining_label`, which has no time separation at all.

**The outcome I predict is continuous first:** `lost_impr_30d` = the page's prior per-30-day run rate minus
what it actually got in the label window. Negative means it grew. Run rate over two blocks instead of one,
because a single 30-day block is jumpy — `impressions_prev_30d` is zero for 3,388 rows.

**Then I threshold it for evaluation:**

```
is_material_decline = 1  if  prior_rate_30d >= 500  and  drop_pct <= -20
```

- The **−20% line is borrowed** from the repo's own `trend_direction` rule, so my label means the same thing
  the reference label means — just measured over a window the features can't see.
- The **500/month floor is a guess.** It's a materiality judgment: a page averaging 30 impressions a month
  doesn't deserve an editor's afternoon, no matter what it did this month. Section 5 shows what happens when I
  move it.
- Both quantities are **observed**; the thresholds are mine. That is the honest version of "defined," and I'll
  keep saying so.

### What the fix costs me

This is the part I didn't expect. **24 of the 44 columns overlap the label window** and are therefore out as
features — every `*_90d` total, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`,
`days_with_impressions`, `impression_tier`, `position_tier`, and of course `trend_pct` / `trend_direction`.
`impression_tier` and `position_tier` are the ones that hurt, because those were ML-01's strongest features.
What survives is 18 numeric columns and 6 categorical ones, built on the two prior blocks. That's 20 surviving
source columns out of 44, turning into 24 model inputs once I add the missingness flags and recompute the
rates on the prior window.

I can rebuild the safe part of the totals by subtraction — `clicks_90d − clicks_last_30d` is a clean 60-day
click count — but not the ones with no `_last_30d` twin (`engaged_sessions_90d`, `pageviews_90d`,
`ai_sessions_90d`) and not the 0–90 day counts, whose composition I can't unpick.

One more trap I avoided on purpose: missingness in this slice follows `content_type`, so a blind `fillna(0)`
on `word_count` or `search_volume` smuggles the content type into the features. I fill zeros **and** keep a
`has_`-flag for each imputed column, so the model can see "unmeasured" instead of guessing it from a zero.

### What this proxy still can't do

- The label window is **not in the future relative to the export** — it's the same 90 days, re-sliced. Both
  windows sit inside one season, so I have no seasonality control. On the warehouse release (weeks 3+) the
  same code shape gets a genuinely forward window, and that re-run is what has to earn the result.
- I'm predicting **lost impressions**, not "declined" as a state. Those aren't the same thing, and I expect to
  revisit the definition once I can see a real forward window in ML-04.

In [3]:
# Split the single 90-day window into three 30-day blocks, then define the target
# on the newest block only. Nothing downstream is allowed to touch the label window.
impr_oldest_30d = (df["impressions_90d"] - df["impressions_prev_30d"] - df["impressions_last_30d"]).astype(float)
impr_prior_30d = df["impressions_prev_30d"].astype(float)
impr_label_30d = df["impressions_last_30d"].astype(float)

impr_prior_60d = impr_oldest_30d + impr_prior_30d          # feature window: days 31-90 back
prior_rate_30d = impr_prior_60d / 2.0                     # per-30-day run rate before the label window
drop_pct = pd.Series(
    np.where(prior_rate_30d > 0,
             (impr_label_30d - prior_rate_30d) / prior_rate_30d.replace(0, np.nan) * 100,
             np.nan),
    index=df.index,
)
lost_impr_30d = prior_rate_30d - impr_label_30d

print("block check - the recovered oldest block should never be negative:")
print(f"  negatives: {int((impr_oldest_30d < 0).sum())}   zeros: {int((impr_oldest_30d == 0).sum())}"
      f" ({100 * (impr_oldest_30d == 0).mean():.1f}% - page had no history that far back)")
print(f"  median {impr_oldest_30d.median():,.0f} | p90 {impr_oldest_30d.quantile(0.9):,.0f} | max {impr_oldest_30d.max():,.0f}")
print(f"  prior_30d zeros: {int((impr_prior_30d == 0).sum())}  <- why I average two blocks, not one")
print()
print("drop_pct (label window vs the prior run rate), rows with a prior rate of 0 are NaN:")
print(drop_pct.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_string())
print(f"  NaN: {int(drop_pct.isna().sum()):,}")
print()

# The review pool: pages that were still getting real demand BEFORE the label window.
POOL_FLOOR = 200          # prior-60d impressions; same demand bar as ML-02, measured in the prior window
VOLUME_FLOOR = 500        # GUESS: a materiality floor, impressions per 30 days before the label window
DROP_PCT_FLOOR = -20      # borrowed from the repo's own trend_direction rule
in_pool = impr_prior_60d >= POOL_FLOOR
is_material_decline = (prior_rate_30d >= VOLUME_FLOOR) & (drop_pct <= DROP_PCT_FLOOR)
is_any_decline = drop_pct <= DROP_PCT_FLOOR
POOL_LOST_IMPRESSIONS = float(lost_impr_30d[in_pool].clip(lower=0).sum())

print(f"review pool (prior-60d >= {POOL_FLOOR}): {int(in_pool.sum()):,} rows = {in_pool.mean() * 100:.1f}% of the slice")
for name, mask in [("any decline <= -20%", is_any_decline),
                   (f"material: >= {VOLUME_FLOOR}/mo AND <= {DROP_PCT_FLOOR}%", is_material_decline)]:
    inside = mask[in_pool]
    print(f"  {name:42s} {int(inside.sum()):6,}  = {inside.mean() * 100:5.2f}% of the pool")
print()
print("how the reference label and mine overlap (pool only):")
ref_down = df["trend_direction"].eq("down")
for name, mine in [("mine", is_material_decline), ("theirs", ref_down)]:
    print(f"  {name:8s} inside pool: {int((mine & in_pool).sum()):6,}  = {100 * (mine & in_pool).sum() / in_pool.sum():.1f}%")
print(f"  both: {int((is_material_decline & ref_down & in_pool).sum()):,}"
      f" | mine only: {int((is_material_decline & ~ref_down & in_pool).sum()):,}"
      f" | theirs only: {int((~is_material_decline & ref_down & in_pool).sum()):,}")
print()
print("total 30-day impressions the pool is losing: "
      f"{POOL_LOST_IMPRESSIONS:,.0f}")

block check - the recovered oldest block should never be negative:
  negatives: 0   zeros: 2598 (8.7% - page had no history that far back)
  median 308 | p90 4,868 | max 258,505
  prior_30d zeros: 3388  <- why I average two blocks, not one

drop_pct (label window vs the prior run rate), rows with a prior rate of 0 are NaN:
count    28361.0
mean       -14.8
std        440.8
min       -100.0
10%        -97.2
25%        -76.3
50%        -47.8
75%         -9.1
90%         43.2
max      42900.0
  NaN: 1,639

review pool (prior-60d >= 200): 19,095 rows = 63.6% of the slice
  any decline <= -20%                        13,307  = 69.69% of the pool
  material: >= 500/mo AND <= -20%             8,407  = 44.03% of the pool

how the reference label and mine overlap (pool only):
  mine     inside pool:  8,407  = 44.0%
  theirs   inside pool: 11,712  = 61.3%
  both: 7,061 | mine only: 1,346 | theirs only: 4,651

total 30-day impressions the pool is losing: 19,472,902


In [4]:
# Why trend_direction isn't one of my features: my label is a rule on trend_direction, so a
# model that can see it is reading its own answer. This is what that looks like. I checked
# whether the reference pipeline does the same thing, and on the evidence in the scripts it
# doesn't - see Section 2.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

y_ref = df["trend_direction"].eq("down").astype(int)
X_only_direction = pd.get_dummies(df[["trend_direction"]].astype(str), dtype=float)
cut = np.arange(len(df))
train_idx, test_idx = cut[:24000], cut[24000:]

probe = LogisticRegression(max_iter=1000)
probe.fit(X_only_direction.iloc[train_idx], y_ref.iloc[train_idx])
probe_scores = probe.predict_proba(X_only_direction.iloc[test_idx])[:, 1]

print("model whose ONLY input is a one-hot of trend_direction:")
print(f"  ROC-AUC on held-out rows: {roc_auc_score(y_ref.iloc[test_idx], probe_scores):.3f}")
print()

# And the columns my own label window takes out of bounds. Anything that counts the
# newest 30 days is unusable as a feature for a label measured on the newest 30 days.
label_window_contaminated = [
    "impressions_90d", "clicks_90d", "sessions_90d", "pageviews_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "impression_tier", "position_tier",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "trend_pct", "trend_direction", "provider_used", "model_used",
]
survivors = [c for c in df.columns if c not in label_window_contaminated]
print(f"columns in the starter CSV: {df.shape[1]}")
print(f"excluded because they overlap the label window: {len(label_window_contaminated)}")
print(f"survivors I can build features from: {len(survivors)}")
print("  " + ", ".join(survivors))
print()
print("two of those exclusions hurt: impression_tier and position_tier were ML-01's")
print("top features, and avg_position was #3 by importance. They are all 90-day")
print("quantities with no _last_30d twin, so there is no way to subtract the label")
print("window out of them. I rebuild demand tiers from the prior 60 days instead.")
print()
print("'excluded' means 'not usable as-is', not 'unrecoverable': clicks_90d, sessions_90d")
print("and impressions_90d each DO have a _last_30d twin, so the prior-60d version is")
print("recoverable by subtraction. That is where my 60-day click and session counts")
print("come from.")

model whose ONLY input is a one-hot of trend_direction:
  ROC-AUC on held-out rows: 1.000

columns in the starter CSV: 44
excluded because they overlap the label window: 24
survivors I can build features from: 20
  content_id, client_id, search_volume, competition, competition_level, cpc, content_type, main_intent, word_count, char_count, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d, content_age_days, age_tier, age_tier_order, days_since_last_update, freshness_tier, word_count_tier, char_count_tier

two of those exclusions hurt: impression_tier and position_tier were ML-01's
top features, and avg_position was #3 by importance. They are all 90-day
quantities with no _last_30d twin, so there is no way to subtract the label
window out of them. I rebuild demand tiers from the prior 60 days instead.

'excluded' means 'not usable as-is', not 'unrecoverable': clicks_90d, sessions_90d
and impressions_90d each DO have a _last_30d twin, so the prior-60d version is
recoverable by subt

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary: Precision@K. K = 50 by default, and K is a capacity input, not something I tune.**

The reason is the ML-02 framing. The reviewer has a fixed number of pages per week. That makes the decision a
set-selection problem, and the honest question is narrow: *of the pages I hand over, how many were worth
handing over?* That's precision inside the top K. ROC-AUC answers a different question (can you separate the
classes at all) and it's the wrong one here, because a ranker with a great AUC and a bad top-50 is useless to
an editor with an afternoon.

K = 50 rather than 40 because it's roughly a week's queue at the capacity I guessed, and because 50 is the K
the repo's own pipeline selects on, which makes my numbers comparable to the reference report's. I print
K = 40, 50 and 200 so nothing in the conclusion rests on one value.

**Secondary, and this is the one I trust more: share of the pool's lost 30-day impressions captured in the top
K.** It has no threshold in it at all — no −20%, no 500/month — so it's the check on whether my label choices
are doing the work. It's also the number an editor actually feels: *the fifty pages on your list account for
X% of what this pool is losing every month.*

**What I won't report:** accuracy at a 0.5 threshold (there is no 0.5 in this workflow), and AUC as a headline
number rather than a diagnostic. On this data AUC shifts by double digits depending on which cut of the label
you take — logistic goes 0.845 → 0.778, the forest 0.895 → 0.714 — while logistic's P@50 barely budges
(0.92 → 0.90). A metric that unstable across label definitions is a diagnostic, not a scoreboard. (The forest's
P@50 does move, 0.96 → 0.78, which is its own problem and shows up again in Section 5.)

**One thing I want on the record now, before Section 5 shows it:** the two metrics disagree, and they can both
be right. P@K asks "were these pages worth reviewing", counting a 900-impression page and a 90,000-impression
page as one each. Loss coverage asks "how much bleeding did I stop", counting them by size. On this slice the
ranking that wins P@K is *not* the ranking that wins coverage. I don't resolve that here — it's a real metric
choice for ML-04/ML-07 — but I'm not going to quietly report whichever one flatters the model.

In [5]:
# The null model for each metric, so later numbers have something to be measured against.
POOL_SIZE = int(in_pool.sum())
BASE_RATE = float(is_material_decline[in_pool].mean())

print(f"pool rows: {POOL_SIZE:,}")
print(f"base rate (a random page is a material decline): {BASE_RATE:.4f}")
print(f"the whole pool is losing {POOL_LOST_IMPRESSIONS:,.0f} impressions per 30 days")
print()

print("Precision@K under a random queue, expected value and spread:")
noise_at_k50 = None
for k in (40, 50, 200, 1000):
    expected = BASE_RATE
    sd = float(np.sqrt(BASE_RATE * (1 - BASE_RATE) / k))   # binomial sd of a single K-page draw
    if k == 50:
        noise_at_k50 = sd
    print(f"  K={k:<5d} expected P@K {expected:.3f}   sd of one draw +/- {sd:.3f}")
print()
print(f"  ^ this is why I never quote a P@50 from a single random draw: at K=50 the")
print(f"    sd is +/- {noise_at_k50:.3f}, so a swing of +/- {2 * noise_at_k50:.2f} is ordinary")
print("    noise rather than signal. Real rankings get compared against the expected")
print("    value, not against one lucky shuffle.")
print()
print("loss coverage @K under a random queue is just K / pool size, because a random")
print("page picks up its share of the loss in expectation:")
for k in (40, 50, 200, 1000):
    print(f"  K={k:<5d} expected loss coverage {k / POOL_SIZE * 100:.2f}%")
print()
print(f"and the capacity arithmetic that sets K: 50 pages is {50 / POOL_SIZE * 100:.2f}% of the pool.")
print("The metric only has room to work in that first half-percent, which is the point.")

pool rows: 19,095
base rate (a random page is a material decline): 0.4403
the whole pool is losing 19,472,902 impressions per 30 days

Precision@K under a random queue, expected value and spread:
  K=40    expected P@K 0.440   sd of one draw +/- 0.078
  K=50    expected P@K 0.440   sd of one draw +/- 0.070
  K=200   expected P@K 0.440   sd of one draw +/- 0.035
  K=1000  expected P@K 0.440   sd of one draw +/- 0.016

  ^ this is why I never quote a P@50 from a single random draw: at K=50 the
    sd is +/- 0.070, so a swing of +/- 0.14 is ordinary
    noise rather than signal. Real rankings get compared against the expected
    value, not against one lucky shuffle.

loss coverage @K under a random queue is just K / pool size, because a random
page picks up its share of the loss in expectation:
  K=40    expected loss coverage 0.21%
  K=50    expected loss coverage 0.26%
  K=200   expected loss coverage 1.05%
  K=1000  expected loss coverage 5.24%

and the capacity arithmetic that sets K

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (`content_id`), belonging to one client. Not a query, not a client-week, not a
page-day — the starter CSV has no page-day grain at all, so any per-day unit is a warehouse-only idea.

- 30,000 rows × 44 columns, 32 clients, and `content_id` is unique, so the grain is 1:1 with pages. No
  dedup needed (the reference `01_prepare_features.py` drops duplicates as a precaution; there are none).
- The **modeling frame** is the review pool: **19,095 rows (63.6% of the slice), spanning 29 of the 32
  clients.** Three clients have no page that clears the prior-window demand bar.
- The frame splits cleanly in two, and I keep the split visible in the column names so I can't accidentally
  train on the outcome: everything left of `prior_rate_30d` is feature-side (built only from days 31–90 back),
  everything from `prior_rate_30d` right is target-side.
- Missing columns are filled with zeros **and** carry a `has_`-flag, because missingness in this slice follows
  `content_type` — a blind `fillna(0)` would encode the content type into the features.
- `content_id` and `client_id` are pseudonyms. They are grouping keys — the client-grouped cross-validation in
  Section 5 uses the second one — and they never enter the feature matrix.
- The frame below is **sorted by lost impressions so the `head()` is readable.** That is a display choice, not
  the queue. The queue's order comes from the model.

The honest caveat on the unit: a "page" here is a row in a teaching export, and I have no way to check whether
one client's `content_id` means the same thing as another's. On the warehouse release the page identity comes
out of `dim_content`, which is the place to settle it.

In [6]:
# The modeling frame, built from the real CSV. One row = one content page.
FEATURE_SIDE = [
    "log_impr_prior_60d", "log_impr_prior_30d", "log_impr_oldest_30d",
    "log_clicks_prior_60d", "log_clicks_prior_30d", "log_sessions_prior_60d", "log_sessions_prior_30d",
    "ctr_prior_60d", "days_since_last_update", "content_age_days",
    "word_count", "char_count", "search_volume", "competition", "cpc",
    "has_word_count", "has_search_volume", "has_competition",
]
CATEGORICAL_SIDE = ["age_tier", "freshness_tier", "content_type", "main_intent",
                    "competition_level", "word_count_tier"]
TARGET_SIDE = ["prior_rate_30d", "impr_label_30d", "drop_pct", "lost_impr_30d", "is_material_decline"]
DISPLAY = [
    "content_id", "client_id", "log_impr_prior_60d", "log_impr_prior_30d", "log_impr_oldest_30d",
    "ctr_prior_60d", "days_since_last_update", "word_count", "age_tier", "content_type",
    "prior_rate_30d", "impr_label_30d", "drop_pct", "lost_impr_30d", "is_material_decline",
]

# Missingness follows content_type in this slice, so I fill zeros but keep an explicit
# has_-flag for every column I impute. A blind fillna(0) smuggles content_type into the features.
word_count_num = pd.to_numeric(df["word_count"], errors="coerce")
char_count_num = pd.to_numeric(df["char_count"], errors="coerce")
search_volume_num = pd.to_numeric(df["search_volume"], errors="coerce")
competition_num = pd.to_numeric(df["competition"], errors="coerce")
cpc_num = pd.to_numeric(df["cpc"], errors="coerce")

frame = pd.DataFrame({
    "content_id": df["content_id"],
    "client_id": df["client_id"],
    "log_impr_prior_60d": np.log1p(impr_prior_60d),
    "log_impr_prior_30d": np.log1p(impr_prior_30d),
    "log_impr_oldest_30d": np.log1p(impr_oldest_30d),
    "log_clicks_prior_60d": np.log1p(df["clicks_90d"] - df["clicks_last_30d"]),
    "log_clicks_prior_30d": np.log1p(df["clicks_prev_30d"]),
    "log_sessions_prior_60d": np.log1p(df["sessions_90d"] - df["sessions_last_30d"]),
    "log_sessions_prior_30d": np.log1p(df["sessions_prev_30d"]),
    "ctr_prior_60d": np.where(impr_prior_60d > 0,
                              (df["clicks_90d"] - df["clicks_last_30d"]) / impr_prior_60d.replace(0, np.nan) * 100,
                              0.0),
    "days_since_last_update": df["days_since_last_update"],
    "content_age_days": df["content_age_days"],
    "word_count": word_count_num.fillna(0.0),
    "char_count": char_count_num.fillna(0.0),
    "search_volume": search_volume_num.fillna(0.0),
    "competition": competition_num.fillna(0.0),
    "cpc": cpc_num.fillna(0.0),
    "has_word_count": word_count_num.notna().astype(int),
    "has_search_volume": search_volume_num.notna().astype(int),
    "has_competition": competition_num.notna().astype(int),
    **{c: df[c].fillna("unknown").astype(str) for c in CATEGORICAL_SIDE},
    "prior_rate_30d": prior_rate_30d,
    "impr_label_30d": impr_label_30d,
    "drop_pct": drop_pct,
    "lost_impr_30d": lost_impr_30d,
    "is_material_decline": is_material_decline,
})

review_pool = frame[in_pool].sort_values("lost_impr_30d", ascending=False).reset_index(drop=True)
print(f"frame built from all rows: {frame.shape}")
print(f"review pool (the unit the queue ranks): {review_pool.shape}")
print(f"  unique content_id: {review_pool['content_id'].nunique():,} "
      f"(duplicates: {int(review_pool['content_id'].duplicated().sum())})")
print(f"  clients represented: {review_pool['client_id'].nunique()} of {df['client_id'].nunique()}")
print(f"  feature-side: {len(FEATURE_SIDE)} numeric + {len(CATEGORICAL_SIDE)} categorical"
      f" | target-side: {', '.join(TARGET_SIDE)}")
print()
print("first 6 rows of the review pool, biggest loss first (display order only,")
print(f"printed as a {len(DISPLAY)}-column subset of the {review_pool.shape[1]}-column frame):")
print(review_pool[DISPLAY].head(6).to_string())

frame built from all rows: (30000, 31)
review pool (the unit the queue ranks): (19095, 31)
  unique content_id: 19,095 (duplicates: 0)
  clients represented: 29 of 32
  feature-side: 18 numeric + 6 categorical | target-side: prior_rate_30d, impr_label_30d, drop_pct, lost_impr_30d, is_material_decline

first 6 rows of the review pool, biggest loss first (display order only,
printed as a 15-column subset of the 31-column frame):
             content_id          client_id  log_impr_prior_60d  log_impr_prior_30d  log_impr_oldest_30d  ctr_prior_60d  days_since_last_update  word_count age_tier     content_type  prior_rate_30d  impr_label_30d   drop_pct  lost_impr_30d  is_material_decline
0  content_8c19996aa890  client_4e07408562           12.947510           11.990928            12.462674       0.127207                      20      2895.0     365+  keyword article        209894.5         89463.0 -57.377159       120431.5                 True
1  content_4c36c775b818  client_4e07408562       

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

I'll answer this with a measurement rather than a claim, because "many signals" is exactly the kind of thing
that turns out to be false. So: five scorers, one pool, one metric, client-grouped cross-validation so no
client appears in both the training folds and its own test fold.

**What an if-statement is up against.** 24 input columns (18 numeric, 6 categorical — 43 columns after
one-hot), and **13 of those 43 have |Spearman| ≥ 0.1 against the label.** The top of that list is volume in
all three windows, clicks, sessions, word count and age. The reference rule in
`scripts/02_baseline_score.py` gets to use four hand-set weights and seven if-conditions. I also wrote the
rule a person would actually write — "stale over 180 days, big, and thin" — to give the if-statement its best
shot.

**The result.** Logistic regression P@50 = 0.92, random forest 0.96, my three-condition rule 0.76, the
repo-style weighted score **0.34 — worse than a random queue at 0.56.** That last one is the most interesting
row: the published rule isn't weak, it's aimed at the wrong thing. It scores visibility and freshness, and
visibility is barely the question. So there is real room between a rule and a model here — 16 to 20 points of
precision at the top of the list, two to three times the ±0.07 noise band from Section 3.

**Three caveats I have to write down, or the table above is misleading:**

1. **The rule that looks perfect is cheating.** Score pages by `impressions_prev_30d / impressions_last_30d` —
   the momentum between the two newest blocks — and on the *any-decline* cut of the label it hits AUC 0.915
   and P@50 = 1.000. That is not skill, it's the answer read backwards: the ratio's denominator is the label
   window. I put it in the table on purpose. Any scorer that approaches 1.0 deserves a look at what it's
   allowed to see before it deserves a citation.

2. **How much of the 0.92 is real?** A fair amount of the material label is page size, and size is easy.
   `log_impr_oldest_30d` has Spearman 0.544 against the material label but **0.022** against the any-decline
   label. Strip the size condition out and the honest signal from prior-window features alone drops to AUC
   0.778 / P@50 0.900 on a 69.7% base rate — still clear of every rule, but no longer a
   number I'd repeat without the caveat attached. A size rule gets about halfway (P@50 = 0.52 on the material
   cut), so "big pages that dipped" is genuinely most of the top of the queue.

3. **P@K and loss coverage disagree, and I'm not picking the flattering one.** Biggest-pages-first captures
   **14.0%** of the pool's lost impressions in its top 200; logistic regression gets 10.3% and the forest 1.5%.
   So the model that wins on precision-at-K is *worse* than a trivial rule at stopping the bleeding on big
   pages. Both are defensible answers to different questions. If the editor's brief is "the biggest losses
   first," selecting on P@K alone builds the wrong queue. This is the metric decision I flagged in Section 3,
   and it stays open for ML-04/ML-07.

**Does the headline survive my own thresholds?** Mostly. The forest holds P@50 between 0.96 and 0.98 anywhere
from a −10% to a −30% drop line, and only falls to 0.80 at −50%. The volume floor is safe anywhere from 200 to
1,000/month — at 2,000/month the forest drops to 0.50 while logistic still holds 0.90. So the *conclusion*
isn't on a knife edge, but the forest is closer to one than I'd like, and the −10%/−30% plateau is suspicious
in the other direction: a metric that barely moves when I tighten or loosen the label is a metric that's partly
just ranking by size.

**What I actually think this shows.** Not "ML works." It shows that on this slice, with a proxy label carved
out of one 90-day window, a learned ranker beats a hand-written rule at the top of the list by enough to be
worth the machinery — and it shows two ways I could fool myself (reading the answer, and mistaking page size
for insight) that I'd never have found by looking at a single headline number. The 0.92 has to be re-earned on
the warehouse release with a genuinely forward window before I believe it. Treat this as *the framing is
workable*, not *the model works*. The direction agrees with ML-01's reference report (rule 0.240 → model
0.740 at P@50), though the two aren't comparable — different label, different split — and I won't put them side
by side as if they were.

In [7]:
# Rule vs model, on the pool, with the label window held out of every feature.
# Client-grouped 5-fold CV: a client is either in a training fold or in its own test fold.
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pool_mask = in_pool.to_numpy()
X_pool = (frame[in_pool]
          .drop(columns=["content_id", "client_id"] + TARGET_SIDE + ["in_review_pool"], errors="ignore")
          .pipe(lambda f: pd.get_dummies(f, columns=CATEGORICAL_SIDE, dtype=float))
          .replace([np.inf, -np.inf], np.nan)
          .fillna(0)
          .reset_index(drop=True))
y_pool = is_material_decline[in_pool].astype(int).reset_index(drop=True)
groups = frame.loc[in_pool, "client_id"].reset_index(drop=True)
loss_pool = lost_impr_30d[in_pool].clip(lower=0).to_numpy()
volume_pool = impr_prior_60d[in_pool].to_numpy()
words_pool = pd.to_numeric(frame.loc[in_pool, "word_count"], errors="coerce").fillna(0).to_numpy()
stale_pool = frame.loc[in_pool, "days_since_last_update"].to_numpy()
prior_30_pool, label_30_pool = impr_prior_30d[in_pool].to_numpy(), impr_label_30d[in_pool].to_numpy()

print(f"feature matrix {X_pool.shape} = {len(FEATURE_SIDE)} numeric + "
      f"{X_pool.shape[1] - len(FEATURE_SIDE)} one-hot")
rho = X_pool.corrwith(y_pool, method="spearman").dropna()
print(f"features with |Spearman| >= 0.1 against the label: {int((rho.abs() >= 0.1).sum())} of {len(rho)}")
print("  " + ", ".join(f"{k} ({v:+.2f})" for k, v in rho.abs().sort_values(ascending=False).head(6).items()))
print()


def precision_at_k(y_true, scores, k):
    top = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return float(np.asarray(y_true)[top].mean())


def loss_coverage_at_k(scores, losses, k):
    top = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return float(np.asarray(losses)[top].sum() / np.asarray(losses).sum())


def percentile_rank(values):
    return pd.Series(np.asarray(values, dtype=float)).rank(method="average", pct=True).to_numpy()


def grouped_oof_scores(y_target, x_frame):
    splitter = GroupKFold(n_splits=5)
    makers = {
        "model: logistic regression": lambda: Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42))]),
        "model: random forest": lambda: RandomForestClassifier(
            n_estimators=300, min_samples_leaf=40, max_depth=12,
            class_weight="balanced_subsample", random_state=42, n_jobs=-1),
    }
    out = {}
    for name, make in makers.items():
        scores = np.zeros(len(x_frame))
        for tr, te in splitter.split(x_frame, y_target, groups=groups):
            model = make()
            model.fit(x_frame.iloc[tr], y_target.iloc[tr])
            scores[te] = model.predict_proba(x_frame.iloc[te])[:, 1]
        out[name] = scores
    return out


models = grouped_oof_scores(y_pool, X_pool)

# The four rules. Two are the ones a person would plausibly write; two are there to be compared against.
visibility = percentile_rank(np.log1p(volume_pool))
freshness = percentile_rank(stale_pool)
depth_gap = (1 - percentile_rank(words_pool)) * visibility
momentum = (pd.Series(np.where(label_30_pool > 0,
                               prior_30_pool / np.where(label_30_pool == 0, np.nan, label_30_pool), np.nan))
            .replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy())

scorers = {
    "random order": np.random.default_rng(0).random(len(y_pool)),
    "rule: biggest pages first": volume_pool,
    "rule: repo-style weighted score": 0.60 * visibility + 0.30 * freshness + 0.10 * depth_gap,
    "rule: 3 if-statements": ((stale_pool >= 180).astype(float)
                              + (volume_pool >= 1000).astype(float)
                              + ((words_pool > 0) & (words_pool < 1200)).astype(float)),
    "rule: momentum (peeks at label window)": momentum,
    **models,
}

results = pd.DataFrame([{
    "scorer": name,
    "AUC": round(roc_auc_score(y_pool, scores), 3),
    "P@40": round(precision_at_k(y_pool, scores, 40), 3),
    "P@50": round(precision_at_k(y_pool, scores, 50), 3),
    "P@200": round(precision_at_k(y_pool, scores, 200), 3),
    "loss cov @50": round(loss_coverage_at_k(scores, loss_pool, 50), 3),
    "loss cov @200": round(loss_coverage_at_k(scores, loss_pool, 200), 3),
} for name, scores in scorers.items()])

print(f"pool {len(y_pool):,} rows | {int(y_pool.sum()):,} material declines | base rate {y_pool.mean():.3f}")
print(f"pool losing {POOL_LOST_IMPRESSIONS:,.0f} impressions per 30 days | "
      f"5 client-grouped folds over {groups.nunique()} clients")
print()
print(results.to_string(index=False))

feature matrix (19095, 43) = 18 numeric + 25 one-hot
features with |Spearman| >= 0.1 against the label: 13 of 43
  log_impr_oldest_30d (+0.54), log_impr_prior_60d (+0.51), log_impr_prior_30d (+0.43), log_clicks_prior_60d (+0.30), log_sessions_prior_60d (+0.27), log_clicks_prior_30d (+0.22)



pool 19,095 rows | 8,407 material declines | base rate 0.440
pool losing 19,472,902 impressions per 30 days | 5 client-grouped folds over 29 clients

                                scorer   AUC  P@40  P@50  P@200  loss cov @50  loss cov @200
                          random order 0.499 0.525  0.56  0.475         0.003          0.013
             rule: biggest pages first 0.797 0.475  0.52  0.470         0.063          0.140
       rule: repo-style weighted score 0.769 0.325  0.34  0.460         0.024          0.060
                 rule: 3 if-statements 0.818 0.775  0.76  0.685         0.005          0.018
rule: momentum (peeks at label window) 0.724 0.200  0.18  0.160         0.001          0.007
            model: logistic regression 0.845 0.950  0.92  0.935         0.040          0.103
                  model: random forest 0.895 0.950  0.96  0.920         0.004          0.015


In [8]:
# Caveats 1 and 2: the cheating rule, and how much of the headline is really just page size.
any_decline_pool = is_any_decline[in_pool].astype(int).reset_index(drop=True)
print(f"any-decline cut of the label (no volume floor): base rate {any_decline_pool.mean():.3f} "
      f"({int(any_decline_pool.sum()):,} of {len(any_decline_pool):,})")
print("the momentum rule uses impressions_prev_30d / impressions_last_30d - its denominator IS")
print("the label window, so it is reading the answer rather than predicting it:")
print()
print(f"  {'scorer':40s} {'AUC':>6s} {'P@50':>6s}")
for name, scores in {
    "random order": np.random.default_rng(0).random(len(any_decline_pool)),
    "rule: biggest pages first": volume_pool,
    "rule: 3 if-statements": scorers["rule: 3 if-statements"],
    "rule: momentum (peeks at label window)": momentum,
    **grouped_oof_scores(any_decline_pool, X_pool),
}.items():
    print(f"  {name:40s} {roc_auc_score(any_decline_pool, scores):6.3f} "
          f"{precision_at_k(any_decline_pool, scores, 50):6.3f}")
print()

# How much of the material label is just 'is this a big page'?
rho_both = pd.DataFrame({
    "rho vs material label": X_pool.corrwith(y_pool, method="spearman"),
    "rho vs any-decline label": X_pool.corrwith(any_decline_pool, method="spearman"),
}).dropna()
print("the same features against both cuts of the label:")
print(rho_both.reindex(rho_both["rho vs material label"].abs()
                       .sort_values(ascending=False).index).head(6).round(3).to_string())
print()
print("volume features carry the material label and are flat against the any-decline label,")
print("so most of the top of the queue is 'big pages that dipped', not a subtle read on decay.")

any-decline cut of the label (no volume floor): base rate 0.697 (13,307 of 19,095)
the momentum rule uses impressions_prev_30d / impressions_last_30d - its denominator IS
the label window, so it is reading the answer rather than predicting it:

  scorer                                      AUC   P@50


  random order                              0.506  0.780
  rule: biggest pages first                 0.462  0.520
  rule: 3 if-statements                     0.487  0.800
  rule: momentum (peeks at label window)    0.915  1.000
  model: logistic regression                0.778  0.900
  model: random forest                      0.714  0.780



the same features against both cuts of the label:
                        rho vs material label  rho vs any-decline label
log_impr_oldest_30d                     0.544                     0.022
log_impr_prior_60d                      0.510                    -0.061
log_impr_prior_30d                      0.430                    -0.159
log_clicks_prior_60d                    0.296                    -0.149
log_sessions_prior_60d                  0.271                    -0.105
log_clicks_prior_30d                    0.215                    -0.203

volume features carry the material label and are flat against the any-decline label,
so most of the top of the queue is 'big pages that dipped', not a subtle read on decay.


In [9]:
# Caveat 3 and the threshold sensitivity: does the headline survive my own choices?
print("drop line sensitivity (pool fixed, volume floor fixed at %d/mo):" % VOLUME_FLOOR)
for drop_floor in (-10, -20, -30, -50):
    alt = ((prior_rate_30d >= VOLUME_FLOOR) & (drop_pct <= drop_floor))[pool_mask].astype(int).to_numpy()
    print(f"  drop <= {drop_floor:>4}%  base {alt.mean():.3f} | "
          f"repo-rule P@50 {precision_at_k(alt, scorers['rule: repo-style weighted score'], 50):.3f} | "
          f"if-stmt {precision_at_k(alt, scorers['rule: 3 if-statements'], 50):.3f} | "
          f"logit {precision_at_k(alt, models['model: logistic regression'], 50):.3f} | "
          f"forest {precision_at_k(alt, models['model: random forest'], 50):.3f}")
print()
print("volume-floor sensitivity (drop line fixed at %d%%):" % DROP_PCT_FLOOR)
for volume_floor in (200, 500, 1000, 2000):
    alt = ((prior_rate_30d >= volume_floor) & (drop_pct <= DROP_PCT_FLOOR))[pool_mask].astype(int).to_numpy()
    print(f"  >= {volume_floor:>5d}/mo  base {alt.mean():.3f} | "
          f"repo-rule P@50 {precision_at_k(alt, scorers['rule: repo-style weighted score'], 50):.3f} | "
          f"if-stmt {precision_at_k(alt, scorers['rule: 3 if-statements'], 50):.3f} | "
          f"logit {precision_at_k(alt, models['model: logistic regression'], 50):.3f} | "
          f"forest {precision_at_k(alt, models['model: random forest'], 50):.3f}")
print()
print("P@K curve for the logistic ranker, against the %0.3f null:" % BASE_RATE)
curve = [precision_at_k(y_pool, models["model: logistic regression"], k) for k in (10, 40, 50, 100, 200, 500, 1000)]
print("  " + "  ".join(f"K={k}:{v:.3f}" for k, v in zip((10, 40, 50, 100, 200, 500, 1000), curve)))

drop line sensitivity (pool fixed, volume floor fixed at 500/mo):
  drop <=  -10%  base 0.487 | repo-rule P@50 0.580 | if-stmt 0.780 | logit 0.960 | forest 0.980
  drop <=  -20%  base 0.440 | repo-rule P@50 0.340 | if-stmt 0.760 | logit 0.920 | forest 0.960
  drop <=  -30%  base 0.388 | repo-rule P@50 0.180 | if-stmt 0.760 | logit 0.920 | forest 0.960
  drop <=  -50%  base 0.267 | repo-rule P@50 0.080 | if-stmt 0.700 | logit 0.820 | forest 0.800

volume-floor sensitivity (drop line fixed at -20%):
  >=   200/mo  base 0.601 | repo-rule P@50 0.340 | if-stmt 0.760 | logit 0.920 | forest 0.960
  >=   500/mo  base 0.440 | repo-rule P@50 0.340 | if-stmt 0.760 | logit 0.920 | forest 0.960
  >=  1000/mo  base 0.313 | repo-rule P@50 0.340 | if-stmt 0.540 | logit 0.920 | forest 0.940
  >=  2000/mo  base 0.197 | repo-rule P@50 0.340 | if-stmt 0.240 | logit 0.900 | forest 0.500

P@K curve for the logistic ranker, against the 0.440 null:
  K=10:0.800  K=40:0.950  K=50:0.920  K=100:0.910  K=200:0.93

## Self-check

- [x] Every section filled — markdown thinking AND the code that backs it
- [x] Notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] `scripts/` untouched, nothing under `data/` committed
- [x] Committed to my repo under `work/notebooks/`, then the repo URL on the card

### Open questions I'm carrying into ML-04

1. **Which metric actually governs the queue** — precision-at-K or share of lost impressions covered? They
   disagree on this slice and they imply different queues. This is the decision I most need a human opinion on.
2. **Is a 90-day feature aggregate spanning the label window acceptable here?** The reference
   feature set is trend-free as far as I can tell, but most of those features are 90-day totals
   while the label is a rule on the newest 30 days. Not a complaint about their pipeline - it's
   the question that made me build my own prior-window frame, and I'd like to know how it's
   meant to be handled on the warehouse release.
3. **Does the proxy survive a real forward window?** Every number in Section 5 comes from a label carved out
   of the same 90 days as its features. The warehouse release is where that gets tested properly.